# Modelos direto por UF · 3/4 · redes, grafos e modelos de fundação (Fases 4 e 5)

**Antes de rodar (uma vez só)**, em `MyDrive/dengue_V4/` (que já tem `dados/`, `referencias_imdc3/` e os arquivos da competição):
1. suba **`uf_lib.py`** e **`uf_autoarima.R`** (pasta `data_imdc_2026/UF/`);
2. suba a **pasta `UF/dados/` com o nome `dados_uf/`** (5 arquivos `.parquet` + `vizinhanca_uf.npz`);
3. substitua **`comp_modelos.py`** e **`comp_inla.R`** pelas versões novas (de `COMPETICAO/`): elas trazem o INLA não linear e a correção do `val_size`.

Os resultados vão para `MyDrive/dengue_V4/resultados_uf/`. Se a sessão cair, rode de novo: o que já foi feito é pulado.
⚠️ Ao abrir vários notebooks ao mesmo tempo, **espere a célula 1 do primeiro terminar** antes de rodar os outros: sessões simultâneas podem criar duas pastas `resultados_uf` no Drive. Se isso acontecer, renomeie a cópia para `resultados_uf_2`; o `CompUF4` e o `Comp5` leem todas as pastas `resultados_uf*`.

**Temporadas:** 2018–2021 = **afinação** (é onde se escolhe tudo); 2022–2025 = **teste** (S1–S4, olhado uma vez no `CompUF4`).

| Notebook | O que roda | Ambiente | Tempo estimado |
|---|---|---|---|
| `CompUF1_portados` | os 13 modelos da competição, direto por UF | CPU High-RAM | 1–2 h (com INLA) |
| `CompUF2_CPU` | auto.arima / auto.ARIMAX (Python e R), 6 VARs, INLA não linear, dois estágios, GBMs com covariáveis, ablação | CPU High-RAM | 4–6 h |
| `CompUF3_GPU` | AGCRN, Chronos, 6 redes com pooling (com e sem exógenas), STID, 4 redes de grafo, Chronos-2 | GPU L4 | 8–12 h |
| `CompUF4_avaliacao` | escolha em 2018–2021, teste em 2022–2025, ensembles | CPU | ~15 min |

**Ambiente:** GPU **L4**. Dá para dividir em sessões com `RODAR_AGORA`.

| Grupo | Modelos | Ideia |
|---|---|---|
| Portados | `AGCRN_UF`, `Chronos_UF` | os da competição, direto por UF |
| neuralforecast com *pooling* | `TFT_pool_UF`, `NHITS_pool_UF`, `TiDE_pool_UF`, `TSMixerx_pool_UF`, `DeepAR_pool_UF`, `PatchTST_pool_UF` | treinados com 435 regiões + 26 UFs (461 séries), prevendo as UFs; Fourier (futuro), anomalias de clima (passado), nível e população (estáticas); 3 sementes |
| Identidade espaço-temporal | `STID_pool_UF`, `STID_UF` | MLP com embedding da unidade e da semana (empata com GNNs na literatura) |
| neuralforecast sem exógenas | `TFT_pool_semcov_UF`, `NHITS_pool_semcov_UF`, `TiDE_pool_semcov_UF` | as mesmas, só com Fourier: mede quanto as exógenas futuras (ENSO, tanom, panom) ajudam |
| Grafo · T-GCN | `GCNGRU_UF` | GRU por UF + 2 camadas de convolução no grafo de fronteiras das 26 UFs |
| Grafo · A3T-GCN | `A3TGCN_UF` | o mesmo + atenção temporal sobre os estados ocultos (Bai et al. 2021) |
| Grafo · STGCN | `STGCN_UF` | blocos "sanduíche" convolução temporal com GLU → grafo → temporal (Yu et al. 2018) |
| Grafo · MTGNN | `MTGNN_UF` | grafo aprendido (direcionado, top-4) somado ao de fronteiras, mix-hop + convoluções dilatadas (Wu et al. 2020) |
| Fundação | `Chronos2_UF`, `Chronos2_cov_UF` | Chronos-2 zero-shot, sem e com covariáveis (anomalias + SEAS5 + ENSO) |
| Opcional | `TimesFM_UF` | **experimental** (API nova do TimesFM 3, não testada): ligue com `USAR_TIMESFM = True` |

⚠️ As redes do neuralforecast, o Chronos-2 e o TimesFM **não puderam ser testados no PC** (sem GPU). STID e as 4 redes de grafo foram testados com 1 época. Um erro vai para `erros_uf_gpu.csv` e os outros modelos seguem: mande o arquivo que eu corrijo.

In [ ]:
USAR_TIMESFM = False
NOTEBOOK = 'uf_gpu'
PACOTES = 'neuralforecast "chronos-forecasting>=2.0"' + (' timesfm' if USAR_TIMESFM else '')

# @title Drive, arquivos e pacotes
import os, sys, shutil, time, glob, subprocess, json, traceback
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_DRIVE, PASTA_LOCAL, NO_COLAB = '/content/drive/MyDrive/dengue_V4', '/content/uf', True
except ImportError:
    PASTA_DRIVE = PASTA_LOCAL = os.path.abspath('.'); NO_COLAB = False
if NO_COLAB:
    os.makedirs(PASTA_LOCAL, exist_ok=True)
    for item in ('v4_lib.py', 'comp_lib.py', 'comp_modelos.py', 'comp_inla.R', 'uf_lib.py', 'uf_autoarima.R', 'dados',
                 'dados_uf', 'referencias_imdc3'):
        o, d = os.path.join(PASTA_DRIVE, item), os.path.join(PASTA_LOCAL, item)
        assert os.path.exists(o), f'Faltou subir {item} para {PASTA_DRIVE}'
        shutil.copytree(o, d, dirs_exist_ok=True) if os.path.isdir(o) else shutil.copy(o, d)
    if PACOTES:
        r = subprocess.run(f'pip install -q {PACOTES}', shell=True, capture_output=True, text=True)
        print(r.stdout[-400:], r.stderr[-400:])
sys.path.insert(0, PASTA_LOCAL)
PASTA_RES = os.path.join(PASTA_DRIVE, 'resultados_uf')
PASTA_COMP = os.path.join(PASTA_DRIVE, 'resultados_competicao')
os.makedirs(os.path.join(PASTA_RES, 'graficos'), exist_ok=True)
import importlib, v4_lib as V, comp_lib as L, comp_modelos as CM, uf_lib as U
for m in (V, L, CM, U):
    importlib.reload(m)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'nenhuma', '· CPUs:', os.cpu_count())
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
ctx = U.ContextoUF(os.path.join(PASTA_LOCAL, 'dados'), os.path.join(PASTA_LOCAL, 'referencias_imdc3'),
                   os.path.join(PASTA_LOCAL, 'dados_uf'))
TUNING, TESTE, TODAS = U.TUNING, U.TESTE, U.TODAS_UF
M = 500                                  # trajetórias por UF
REFAZER = False

In [ ]:
cfg_agcrn = dict(n_sementes=2, epocas_max=40, paciencia=6, frac_origens=0.5, n_traj=M)
mods = U.portados_gpu(cfg_agcrn)
mods += [U.NFPooled(a) for a in ('TFT', 'NHITS', 'TiDE', 'TSMixerx', 'DeepAR', 'PatchTST')]
mods += [U.NFPooled(a, covariaveis=False) for a in ('TFT', 'NHITS', 'TiDE')]
mods += [U.STID(pool=True), U.STID(pool=False)]
mods += [U.GCNGRU(), U.GCNGRU(atencao=True), U.STGCN(), U.MTGNN()]
mods += [U.Chronos2(False), U.Chronos2(True)]
if USAR_TIMESFM:
    mods.append(U.TimesFM())
MODELOS = {m.nome: m for m in mods}
PLANO_COMPLETO = {m.nome: TODAS for m in mods}
RODAR_AGORA = list(PLANO_COMPLETO)      # ex.: [k for k in PLANO_COMPLETO if 'pool' in k]
PLANO = {k: v for k, v in PLANO_COMPLETO.items() if k in RODAR_AGORA}
print(list(PLANO))

In [ ]:
# @title Rodar (retoma de onde parou; um modelo com erro não interrompe os outros)
erros = []
for nome, temps in PLANO.items():
    for s in temps:
        try:
            dt = L.rodar(ctx, MODELOS[nome], s, PASTA_RES, M=M, refazer=REFAZER)
            if dt is not None:
                print(f'✅ {nome:34s} {s}: {dt / 60:5.1f} min', flush=True)
        except Exception as e:
            erros.append((nome, s, repr(e)))
            print(f'❌ {nome:34s} {s}: {repr(e)[:400]}', flush=True)
            traceback.print_exc(limit=2)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
print('Erros:', len(erros))
pd.DataFrame(erros, columns=['modelo', 'temporada', 'erro']).to_csv(os.path.join(PASTA_RES, f'erros_{NOTEBOOK}.csv'), index=False)
t = L.tempos(PASTA_RES)
t[t.modelo.isin(list(PLANO))].groupby('modelo').segundos.agg(['count', 'mean', 'sum']).div([1, 60, 60]).round(1) \
    .rename(columns={'count': 'temporadas', 'mean': 'min/temporada', 'sum': 'min total'})